# Patient Monitoring Delta Lakehouse

This notebook implements a Medallion Architecture:

Kafka Validated Data
        |
        v
Bronze Layer
(raw validated patient readings)
        |
        v
Silver Layer
(cleaned and deduplicated records)
        |
        v
Gold Layer
(patient health analytics aggregates)

Delta Lake is used for ACID transactions,
schema enforcement, and MERGE operations.

In [2]:
import os
import sys

os.environ["HADOOP_HOME"] = r"C:\hadoop"
os.environ["hadoop.home.dir"] = r"C:\hadoop"

print("HADOOP_HOME =", os.environ.get("HADOOP_HOME"))
print("Python =", sys.executable)

HADOOP_HOME = C:\hadoop
Python = C:\Users\reems\Desktop\patient-monitoring-capstone\.venv\Scripts\python.exe


In [ ]:
import os
import sys

os.environ["PYSPARK_PYTHON"] = sys.executable
os.environ["PYSPARK_DRIVER_PYTHON"] = sys.executable

print(sys.executable)

In [ ]:
from delta import configure_spark_with_delta_pip
from pyspark.sql import SparkSession


builder = (
    SparkSession.builder
    .appName("patient-monitoring-delta")
    .config(
        "spark.sql.extensions",
        "io.delta.sql.DeltaSparkSessionExtension"
    )
    .config(
        "spark.sql.catalog.spark_catalog",
        "org.apache.spark.sql.delta.catalog.DeltaCatalog"
    )
)

spark = configure_spark_with_delta_pip(builder).getOrCreate()

print("Delta Ready")

Py4JJavaError: An error occurred while calling None.org.apache.spark.api.java.JavaSparkContext.
: java.lang.RuntimeException: java.io.FileNotFoundException: java.io.FileNotFoundException: HADOOP_HOME and hadoop.home.dir are unset. -see https://wiki.apache.org/hadoop/WindowsProblems
	at org.apache.hadoop.util.Shell.getWinUtilsPath(Shell.java:735)
	at org.apache.hadoop.util.Shell.getSetPermissionCommand(Shell.java:270)
	at org.apache.hadoop.fs.FileUtil.chmod(FileUtil.java:1139)
	at org.apache.hadoop.fs.FileUtil.chmod(FileUtil.java:1125)
	at org.apache.spark.util.Utils$.fetchFile(Utils.scala:489)
	at org.apache.spark.SparkContext.addFile(SparkContext.scala:1795)
	at org.apache.spark.SparkContext.$anonfun$new$16(SparkContext.scala:533)
	at org.apache.spark.SparkContext.$anonfun$new$16$adapted(SparkContext.scala:533)
	at scala.collection.immutable.List.foreach(List.scala:431)
	at org.apache.spark.SparkContext.<init>(SparkContext.scala:533)
	at org.apache.spark.api.java.JavaSparkC

In [ ]:
from pyspark.sql import SparkSession
from delta import configure_spark_with_delta_pip

from pyspark.sql.functions import *

In [12]:
builder = (
    SparkSession.builder
    .appName("patient-monitoring-delta")
    .config(
        "spark.sql.extensions",
        "io.delta.sql.DeltaSparkSessionExtension"
    )
    .config(
        "spark.sql.catalog.spark_catalog",
        "org.apache.spark.sql.delta.catalog.DeltaCatalog"
    )
)

spark = configure_spark_with_delta_pip(builder).getOrCreate()

print("Spark + Delta Ready")

Py4JJavaError: An error occurred while calling None.org.apache.spark.api.java.JavaSparkContext.
: java.lang.RuntimeException: java.io.FileNotFoundException: java.io.FileNotFoundException: HADOOP_HOME and hadoop.home.dir are unset. -see https://wiki.apache.org/hadoop/WindowsProblems
	at org.apache.hadoop.util.Shell.getWinUtilsPath(Shell.java:735)
	at org.apache.hadoop.util.Shell.getSetPermissionCommand(Shell.java:270)
	at org.apache.hadoop.fs.FileUtil.chmod(FileUtil.java:1139)
	at org.apache.hadoop.fs.FileUtil.chmod(FileUtil.java:1125)
	at org.apache.spark.util.Utils$.fetchFile(Utils.scala:489)
	at org.apache.spark.SparkContext.addFile(SparkContext.scala:1795)
	at org.apache.spark.SparkContext.$anonfun$new$16(SparkContext.scala:533)
	at org.apache.spark.SparkContext.$anonfun$new$16$adapted(SparkContext.scala:533)
	at scala.collection.immutable.List.foreach(List.scala:431)
	at org.apache.spark.SparkContext.<init>(SparkContext.scala:533)
	at org.apache.spark.api.java.JavaSparkC

## Bronze Layer

Bronze layer stores validated raw events coming from the ingestion layer.

Source:
Kafka validated patient readings

Format:
Delta Lake

Purpose:
- Preserve original data
- Enable ACID transactions
- Provide historical record

In [5]:
BRONZE_PATH = "../data/bronze/patient_readings"
SILVER_PATH = "../data/silver/patient_readings"
GOLD_PATH = "../data/gold/patient_summary"

In [7]:
data = [
    ("P001", 82, 98, "2026-07-22T10:00:00"),
    ("P002", 90, 95, "2026-07-22T10:01:00"),
    ("P003", 75, 99, "2026-07-22T10:02:00")
]


columns = [
    "patient_id",
    "heart_rate",
    "oxygen_level",
    "event_time"
]


bronze_df = spark.createDataFrame(
    data,
    columns
)

bronze_df.show()

Py4JJavaError: An error occurred while calling o98.showString.
: org.apache.spark.SparkException: Cannot find catalog plugin class for catalog 'spark_catalog': org.apache.spark.sql.delta.catalog.DeltaCatalog.
	at org.apache.spark.sql.errors.QueryExecutionErrors$.catalogPluginClassNotFoundForCatalogError(QueryExecutionErrors.scala:1925)
	at org.apache.spark.sql.connector.catalog.Catalogs$.load(Catalogs.scala:70)
	at org.apache.spark.sql.connector.catalog.CatalogManager.loadV2SessionCatalog(CatalogManager.scala:67)
	at org.apache.spark.sql.connector.catalog.CatalogManager.$anonfun$v2SessionCatalog$2(CatalogManager.scala:86)
	at scala.collection.mutable.HashMap.getOrElseUpdate(HashMap.scala:86)
	at org.apache.spark.sql.connector.catalog.CatalogManager.$anonfun$v2SessionCatalog$1(CatalogManager.scala:86)
	at scala.Option.map(Option.scala:230)
	at org.apache.spark.sql.connector.catalog.CatalogManager.v2SessionCatalog(CatalogManager.scala:85)
	at org.apache.spark.sql.connector.catal

In [ ]:
bronze_df.write \
    .format("delta") \
    .mode("overwrite") \
    .save(BRONZE_PATH)

print("Bronze created")

In [ ]:
spark.read \
    .format("delta") \
    .load(BRONZE_PATH) \
    .show()